In [1]:
import pandas as pd
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from django.core.exceptions import ObjectDoesNotExist
from apis_core.apis_metainfo.models import Collection, Uri, TempEntityClass
from apis_core.apis_entities.models import Person
from apis_core.apis_vocabularies.models import ProfessionType


In [2]:
col = Collection.objects.get(id=122)
domain = "spielplan-oper_1869–1945"

In [3]:
df = gsheet_to_df("1WxCkQJHbUMzp-nePrlC2V9pu77UiqGoeKphW2gPzlDk")

200


In [4]:
df.head(3)

,pmb-nummer,uri,domain
0,8931,https://www.mdw.ac.at/imi/operapolitics/spielp...,spielplan-oper_1869–1945
1,359598,https://www.mdw.ac.at/imi/operapolitics/spielp...,spielplan-oper_1869–1945
2,6676,https://www.mdw.ac.at/imi/operapolitics/spielp...,spielplan-oper_1869–1945


In [5]:
# adds opern-uris to existing entities
no_match = []
for i, row in tqdm(df.iterrows(), total=len(df)):
    if pd.isna(row["pmb-nummer"]):
        continue
    else:
        pmb_id = int(row["pmb-nummer"])
    try:
        entity = TempEntityClass.objects.get(id=pmb_id)
    except ObjectDoesNotExist:
        try:
            entity = Uri.objects.get(uri=f"https://pmb.acdh.oeaw.ac.at/entity/{row['pmb-nummer']}/").entity
        except ObjectDoesNotExist:
            no_match.append(row["pmb-nummer"])
            continue
    entity.collection.add(col)
    bt_uri, _ = Uri.objects.get_or_create(uri=row["uri"], entity=entity, domain=domain)
for x in no_match:
    print(x)

100%|█████████████████████████████████████████████████████████████████████████████| 981/981 [00:07<00:00, 132.14it/s]


In [6]:
# import new persons
df = gsheet_to_df("1H4D6V_pYDvswBBvSbxv79Et4F9LQO1qnCuIGGR-v484")
df.head(3)

200


,opera_id,uri,domain,surname,forename,name_note,birth_year,death_year,occupations,is_workshop,note
0,1000,https://www.mdw.ac.at/imi/operapolitics/spielp...,spielplan-oper_1869–1945,Leuven,Adolphe de,NaN,1800.0,1844.0,Librettist/Librettistin,0,NaN
1,1005,https://www.mdw.ac.at/imi/operapolitics/spielp...,spielplan-oper_1869–1945,Gheusi,Pierre-Barthélemy,Pseud. Norbert Lorédan,1865.0,1943.0,Librettist/Librettistin,0,NaN
2,1020,https://www.mdw.ac.at/imi/operapolitics/spielp...,spielplan-oper_1869–1945,Pordes,Alexander Siegmund,auch Pordes-Milo,1878.0,1931.0,Librettist/Librettistin,0,NaN


In [7]:
professions = {}
for i, row in tqdm(df.iterrows(), total=len(df)):
    pmb_uri, _ = Uri.objects.get_or_create(uri=row["uri"], domain=domain)
    entity = pmb_uri.entity
    if entity:
        person = Person.objects.get(id=entity.id)
        pass
    else:
        item = {
            "name": row["surname"][:254] if pd.notna(row["surname"]) else "",
            "first_name": row["forename"][:254] if pd.notna(row["forename"]) else "",
        }
        
        entity = Person.objects.create(**item)
        try:
            entity.start_date_written = str(int(row["birth_year"]))
        except:
            pass
        try:
            entity.end_date_written = str(int(row["death_year"]))
        except:
            pass

        entity.save()
        person = Person.objects.get(id=entity.id)
        try:
            job = professions[row["occupations"]]
        except KeyError:
            job = ProfessionType.objects.get_or_create(name=row["occupations"])[0]
            professions[row["occupations"]] = job
        person.profession.add(job)
        pmb_uri.entity = entity
        pmb_uri.save()
    entity.collection.add(col)

100%|██████████████████████████████████████████████████████████████████████████████| 141/141 [00:03<00:00, 45.90it/s]
